# 18 — Manager Analytics Evaluation

Benchmark نهایی بخش Analytics.

ساختار ارزیابی:
- **۱۵ case ثابت**: ۵ DEV + ۱۰ TEST
- Numeric Faithfulness: deterministic
- Aggregation / filter / comparison facts: deterministic و مستقل از LLM judge
- کیفیت توضیح مدیریتی و caveat compliance: LLM-assisted judge
- latency / tokens / cost
- failure analysis
- checkpoint برای جلوگیری از API call تکراری

> نکته: امتیازهای کیفی judge، human gold مستقل نیستند و باید به‌عنوان proxy گزارش شوند.


In [1]:
from pathlib import Path
import json
import os
import sys

import pandas as pd
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

load_dotenv(
    PROJECT_ROOT
    / ".env"
)

from src.rag.config import load_config
from src.rag.evaluation import (
    AnalyticsEvaluationDataset,
    ManagerAnalyticsEvaluator,
    analytics_failure_summary,
    build_analytics_report,
    export_analytics_manual_review,
)
from src.rag.evaluation.analytics_metrics import (
    summarize_analytics_results,
)
from src.rag.evaluation.analytics_runtime import (
    load_analytics_evaluation_context,
)

In [2]:
evaluation_config = load_config(
    PROJECT_ROOT
    / "configs"
    / "analytics_evaluation.yaml"
)[
    "analytics_evaluation"
]

dataset = (
    AnalyticsEvaluationDataset
    .load(
        PROJECT_ROOT
        / evaluation_config[
            "dataset"
        ][
            "path"
        ]
    )
)

display(
    dataset.to_frame()
)

print(
    dataset.to_frame()[
        "split"
    ].value_counts()
)

print()
print(
    dataset.to_frame()[
        "case_type"
    ].value_counts()
)

,case_id,split,case_type,question,filters,comparison_categories,policy_expectations
0,a001,dev,category_overview,وضعیت کلی دسته دفتر را از نظر اندازه کاتالوگ، ...,{'Category2': 'دفتر'},[],[rating_coverage_caveat]
1,a002,dev,price_analysis,از نظر قیمت فعلی، میانه و بازه میانی قیمت دفتر...,{'Category2': 'دفتر'},[],[]
2,a003,dev,category_comparison,دفتر و زیورآلات زنانه و مردانه را از نظر قیمت،...,{},"[دفتر, زیورآلات زنانه و مردانه]",[rating_coverage_caveat]
3,a004,dev,brand_policy,کدام برندها در دسته دفتر غالب‌اند و سهم بازار ...,{'Category2': 'دفتر'},[],[brand_coverage_caveat]
4,a005,dev,historical_price_refusal,قیمت دفترها نسبت به ماه قبل چقدر تغییر کرده و ...,{'Category2': 'دفتر'},[],[historical_price_refusal]
5,a006,test,rating_analysis,کیفیت امتیازدهی محصولات دفتر چطور است و چقدر م...,{'Category2': 'دفتر'},[],[rating_coverage_caveat]
6,a007,test,review_coverage,پوشش بازخورد کاربران در دسته رنگ مو چقدر است و...,{'Category2': 'رنگ مو'},[],[]
7,a008,test,insufficient_review_data,از تجربه کاربران درباره دسته گردنبند طلا زنانه...,{'Category2': 'گردنبند طلا زنانه'},[],[zero_review_awareness]
8,a009,test,category_comparison,دفتر و لباس دخترانه را از نظر اندازه دسته، میا...,{},"[دفتر, لباس دخترانه]",[rating_coverage_caveat]
9,a010,test,category_comparison,رنگ مو و تجهیزات کمپینگ را از نظر پوشش بازخورد...,{},"[رنگ مو, تجهیزات کمپینگ]",[rating_coverage_caveat]


split
test    10
dev      5
Name: count, dtype: int64

case_type
category_comparison         3
brand_policy                2
category_overview           1
price_analysis              1
historical_price_refusal    1
rating_analysis             1
review_coverage             1
insufficient_review_data    1
review_volume_policy        1
category1_overview          1
catalog_overview            1
three_way_comparison        1
Name: count, dtype: int64


## ساخت Context و Scope Validation

این cell قبل از اولین answer/judge API call اجرا می‌شود.
اگر هر Category/Filter benchmark در دیتاست وجود نداشته باشد، اجرا همین‌جا متوقف می‌شود.


In [3]:
context = (
    load_analytics_evaluation_context(
        project_root=(
            PROJECT_ROOT
        ),
        api_key=os.environ[
            "METIS_API_KEY"
        ],
        base_url=os.environ[
            "METIS_BASE_URL"
        ],
    )
)

scope_validation = (
    dataset.validate_scopes(
        context.analytics_service
    )
)

print(
    "Scope validation:",
    scope_validation,
)

print(
    "Canonical analytics products:",
    f"{len(context.analytics_service.products):,}",
)

Scope validation: {'case_count': 15, 'dev_count': 5, 'test_count': 10, 'status': 'ok'}
Canonical analytics products: 948,352


In [4]:
output_config = (
    evaluation_config[
        "output"
    ]
)

OUTPUT_DIR = (
    PROJECT_ROOT
    / output_config[
        "directory"
    ]
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

CHECKPOINT_PATH = (
    OUTPUT_DIR
    / output_config[
        "checkpoint"
    ]
)

evaluator = (
    ManagerAnalyticsEvaluator(
        analytics_pipeline=(
            context.pipeline
        ),
        judge=context.judge,
        dataset=dataset,
        generic_brand_values=(
            context.analytics_config[
                "audit"
            ][
                "generic_brand_values"
            ]
        ),
        rating_max=(
            context.analytics_config[
                "aggregation"
            ][
                "product_rating_max"
            ]
        ),
        weights=(
            evaluation_config[
                "weights"
            ]
        ),
    )
)

print(
    "Checkpoint:",
    CHECKPOINT_PATH,
)

print(
    "Planned answer calls: 15"
)

print(
    "Planned judge calls: 15"
)

Checkpoint: /home/ali/Desktop/projects/digikala-ai-assistant/data/evaluation/analytics/analytics_checkpoint.jsonl
Planned answer calls: 15
Planned judge calls: 15


## اجرای Benchmark

`FORCE_RERUN=False` بماند مگر اینکه عمداً بخواهید همه‌ی API callها دوباره اجرا شوند.


In [5]:
FORCE_RERUN = False

results = evaluator.run(
    checkpoint_path=(
        CHECKPOINT_PATH
    ),
    splits=[
        "dev",
        "test",
    ],
    force=(
        FORCE_RERUN
    ),
)

print()
print(
    "Status:"
)

print(
    results[
        "status"
    ].value_counts(
        dropna=False
    )
)

[1/15] a001: running
[2/15] a002: running
[3/15] a003: running
[4/15] a004: running
[5/15] a005: running
[6/15] a006: running
[7/15] a007: running
[8/15] a008: running
[9/15] a009: running
[10/15] a010: running
[11/15] a011: running
[12/15] a012: running
[13/15] a013: running
[14/15] a014: running
[15/15] a015: running

Status:
status
ok    15
Name: count, dtype: int64


## Aggregate Metrics

In [6]:
summary = (
    summarize_analytics_results(
        results
    )
)

display(
    summary
)

,split,cases,successful_cases,overall_judge_score,numeric_faithfulness,fact_value_accuracy,scope_product_count_accuracy,comparison_fact_accuracy,rendered_metric_accuracy,policy_guard_configuration,...,completeness,relevance,managerial_usefulness,instruction_following,answer_latency_ms,judge_latency_ms,evaluation_latency_ms,answer_total_tokens,judge_total_tokens,answer_latency_p95_ms
0,dev,5,5,4.730000,1.0,1.0,1.0,1.0,1.0,1.0,...,4.800000,5.0,4.8,4.800000,10477.990995,9926.898837,20831.705693,3206.2,2917.2,16385.477017
1,test,10,10,4.840000,1.0,1.0,1.0,1.0,1.0,1.0,...,4.900000,5.0,4.8,4.900000,14493.414109,10397.072602,25403.470421,3537.7,3136.2,20666.823143
2,ALL,15,15,4.803333,1.0,1.0,1.0,1.0,1.0,1.0,...,4.866667,5.0,4.8,4.866667,13154.939738,10240.348014,23879.548845,3427.2,3063.2,19771.165598


### Primary deterministic metrics

این‌ها primary و مستقل از LLM judge هستند:

- `numeric_faithfulness`
- `fact_value_accuracy`
- `scope_product_count_accuracy`
- `comparison_fact_accuracy`
- `rendered_metric_accuracy`


In [7]:
deterministic_columns = [
    "case_id",
    "split",
    "case_type",
    "numeric_faithfulness",
    "fact_value_accuracy",
    "scope_product_count_accuracy",
    "comparison_fact_accuracy",
    "rendered_metric_accuracy",
    "policy_guard_configuration",
]

display(
    results[
        [
            column
            for column
            in deterministic_columns
            if column
            in results.columns
        ]
    ]
)

,case_id,split,case_type,numeric_faithfulness,fact_value_accuracy,scope_product_count_accuracy,comparison_fact_accuracy,rendered_metric_accuracy,policy_guard_configuration
0,a001,dev,category_overview,1.0,1.0,1.0,NaN,1.0,1.0
1,a002,dev,price_analysis,1.0,1.0,1.0,NaN,1.0,NaN
2,a003,dev,category_comparison,1.0,1.0,1.0,1.0,1.0,1.0
3,a004,dev,brand_policy,1.0,1.0,1.0,NaN,1.0,1.0
4,a005,dev,historical_price_refusal,1.0,1.0,1.0,NaN,1.0,1.0
5,a006,test,rating_analysis,1.0,1.0,1.0,NaN,1.0,1.0
6,a007,test,review_coverage,1.0,1.0,1.0,NaN,1.0,NaN
7,a008,test,insufficient_review_data,1.0,1.0,1.0,NaN,1.0,1.0
8,a009,test,category_comparison,1.0,1.0,1.0,1.0,1.0,1.0
9,a010,test,category_comparison,1.0,1.0,1.0,1.0,1.0,1.0


## LLM-assisted Quality Scores

In [8]:
quality_columns = [
    "case_id",
    "split",
    "case_type",
    "overall_judge_score",
    "correctness",
    "groundedness",
    "caveat_compliance",
    "completeness",
    "relevance",
    "managerial_usefulness",
    "instruction_following",
    "judge_failure_tags",
]

display(
    results[
        quality_columns
    ].sort_values(
        [
            "split",
            "overall_judge_score",
        ],
        ascending=[
            True,
            True,
        ],
    )
)

,case_id,split,case_type,overall_judge_score,correctness,groundedness,caveat_compliance,completeness,relevance,managerial_usefulness,instruction_following,judge_failure_tags
3,a004,dev,brand_policy,3.65,3.0,2.0,5.0,4.0,5.0,4.0,4.0,[unsupported_claim]
0,a001,dev,category_overview,5.00,5.0,5.0,5.0,5.0,5.0,5.0,5.0,[]
1,a002,dev,price_analysis,5.00,5.0,5.0,5.0,5.0,5.0,5.0,5.0,[]
2,a003,dev,category_comparison,5.00,5.0,5.0,5.0,5.0,5.0,5.0,5.0,[]
4,a005,dev,historical_price_refusal,5.00,5.0,5.0,5.0,5.0,5.0,5.0,5.0,[]
11,a012,test,brand_policy,3.45,3.0,1.0,5.0,4.0,5.0,4.0,4.0,[unsupported_claim]
10,a011,test,review_volume_policy,4.95,5.0,5.0,5.0,5.0,5.0,4.0,5.0,[]
5,a006,test,rating_analysis,5.00,5.0,5.0,5.0,5.0,5.0,5.0,5.0,[]
6,a007,test,review_coverage,5.00,5.0,5.0,5.0,5.0,5.0,5.0,5.0,[]
7,a008,test,insufficient_review_data,5.00,5.0,5.0,5.0,5.0,5.0,5.0,5.0,[]


## Failure Analysis

In [9]:
failures = (
    analytics_failure_summary(
        results
    )
)

display(
    failures
)

flagged = results[
    results[
        "judge_failure_tags"
    ].apply(
        lambda value: (
            isinstance(
                value,
                list,
            )
            and len(
                value
            )
            > 0
        )
    )
    |
    (
        results[
            "numeric_faithfulness"
        ]
        < 1.0
    )
    |
    (
        results[
            "fact_value_accuracy"
        ]
        < 1.0
    )
]

display(
    flagged[
        [
            "case_id",
            "split",
            "case_type",
            "question",
            "answer",
            "overall_judge_score",
            "judge_failure_tags",
            "judge_summary_reason",
        ]
    ]
)

,failure,count
0,unsupported_claim,2


,case_id,split,case_type,question,answer,overall_judge_score,judge_failure_tags,judge_summary_reason
3,a004,dev,brand_policy,کدام برندها در دسته دفتر غالب‌اند و سهم بازار ...,سهم بازار برندها از این داده قابل محاسبه یا گز...,3.65,[unsupported_claim],پاسخ در رعایت سیاست مهمِ عدم تبدیل سهم کاتالوگ...
11,a012,test,brand_policy,برندهای دسته دفتر را بر اساس سهم بازار رتبه‌بن...,رتبه‌بندی «سهم بازار» و تعیین رهبر بازار از دا...,3.45,[unsupported_claim],پاسخ از نظر سیاست داده و caveat برند بسیار منا...


## Latency / Tokens / Cost

In [10]:
telemetry = pd.DataFrame(
    [
        {
            "split": split,
            "mean_answer_latency_sec": (
                group[
                    "answer_latency_ms"
                ].mean()
                / 1000
            ),
            "p95_answer_latency_sec": (
                group[
                    "answer_latency_ms"
                ].quantile(
                    0.95
                )
                / 1000
            ),
            "mean_eval_latency_sec": (
                group[
                    "evaluation_latency_ms"
                ].mean()
                / 1000
            ),
            "answer_tokens": int(
                group[
                    "answer_total_tokens"
                ].sum()
            ),
            "judge_tokens": int(
                group[
                    "judge_total_tokens"
                ].sum()
            ),
        }
        for split, group
        in results.groupby(
            "split",
            sort=False,
        )
    ]
)

display(
    telemetry
)

answer_cost = pd.to_numeric(
    results[
        "answer_cost_usd"
    ],
    errors="coerce",
)

judge_cost = pd.to_numeric(
    results[
        "judge_cost_usd"
    ],
    errors="coerce",
)

if (
    answer_cost.notna().any()
    or judge_cost.notna().any()
):
    print(
        "Answer cost USD:",
        answer_cost.sum(
            min_count=1
        ),
    )

    print(
        "Judge cost USD:",
        judge_cost.sum(
            min_count=1
        ),
    )
else:
    print(
        "Cost not computed: pricing fields are null."
    )

,split,mean_answer_latency_sec,p95_answer_latency_sec,mean_eval_latency_sec,answer_tokens,judge_tokens
0,dev,10.477991,16.385477,20.831706,16031,14586
1,test,14.493414,20.666823,25.403470,35377,31362


Cost not computed: pricing fields are null.


## Export

In [11]:
CASE_RESULTS_PATH = (
    OUTPUT_DIR
    / output_config[
        "case_results"
    ]
)

METRICS_PATH = (
    OUTPUT_DIR
    / output_config[
        "metrics"
    ]
)

MANUAL_REVIEW_PATH = (
    OUTPUT_DIR
    / output_config[
        "manual_review"
    ]
)

REPORT_PATH = (
    OUTPUT_DIR
    / output_config[
        "report"
    ]
)

serializable = results.copy()

for column in [
    "filters",
    "comparison_categories",
    "policy_expectations",
    "fact_value_errors",
    "judge_failure_tags",
    "result",
    "judge",
]:
    if column in serializable.columns:
        serializable[
            column
        ] = serializable[
            column
        ].apply(
            lambda value: json.dumps(
                value,
                ensure_ascii=False,
                default=str,
            )
        )

serializable.to_csv(
    CASE_RESULTS_PATH,
    index=False,
    encoding="utf-8-sig",
)

summary.to_parquet(
    METRICS_PATH,
    index=False,
)

export_analytics_manual_review(
    frame=results,
    path=(
        MANUAL_REVIEW_PATH
    ),
)

report = (
    build_analytics_report(
        results
    )
)

with REPORT_PATH.open(
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        report,
        handle,
        ensure_ascii=False,
        indent=2,
        default=str,
    )

print(
    "Saved:",
    CASE_RESULTS_PATH,
)

print(
    "Saved:",
    METRICS_PATH,
)

print(
    "Saved:",
    MANUAL_REVIEW_PATH,
)

print(
    "Saved:",
    REPORT_PATH,
)

Saved: /home/ali/Desktop/projects/digikala-ai-assistant/data/evaluation/analytics/analytics_case_results.csv
Saved: /home/ali/Desktop/projects/digikala-ai-assistant/data/evaluation/analytics/analytics_metrics.parquet
Saved: /home/ali/Desktop/projects/digikala-ai-assistant/data/evaluation/analytics/analytics_manual_review.csv
Saved: /home/ali/Desktop/projects/digikala-ai-assistant/data/evaluation/analytics/analytics_report.json


## Reporting caveat

برای گزارش نهایی:

> Manager Analytics numeric faithfulness and deterministic fact accuracy are
> independently checked in Python. Qualitative answer-quality scores use an
> LLM-assisted judge and are therefore proxy evaluation rather than independent
> human annotation.
